In [4]:
# -*- coding: utf-8 -*-
# =====================================================================================
#  [학생용] 결과기 개발 기본 틀 — 1번 셀
# =====================================================================================
#  이 셀은 완성된 결과기가 아닙니다. 1번 셀에 팀별 결과기를 구현한 뒤 사용합니다.
#  결과기 코랩은 아래 두 셀을 위에서 아래로 한 번 실행할 수 있어야 합니다.
#
#    1번 셀: 팀별 결과기 구현 — 이 파일의 코드
#    2번 셀: 공개 10문항 공통 러너 — 운영진 배포본, 팀 식별자 한 줄 외 수정 금지
# =====================================================================================


# -------------------------------------------------------------------------------------
# 0. 고정 기준 — 문서명과 생성 모델 계열
# -------------------------------------------------------------------------------------
OFFICIAL_DOCUMENT_NAMES = (
    "카카오계정 약관",
    "카카오 위치정보 이용약관",
    "카카오 통합서비스약관",
    "카카오 통합 약관",
)
REQUIRED_GENERATION_MODEL_FAMILY = "Qwen2.5-Instruct"


# =====================================================================================
# 1. 팀별 자유 구현 영역 — BM25 + 리랭커 하이브리드 검색 + Qwen2.5 생성
# =====================================================================================
import json as _json
import math as _math
import re as _re
import subprocess as _subprocess
import sys as _sys
import time as _time
import unicodedata as _unicodedata


def _pip_install(*pkgs):
    _subprocess.run([_sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)


_pip_install("rank_bm25", "sentence-transformers")

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("[환경]", _DEVICE, "|", torch.cuda.get_device_name(0) if _DEVICE == "cuda" else "GPU 없음")


# -------------------------------------------------------------------------------------
# 1-1. 약관 원문 — 조 단위, 요약 없이 원문 그대로. 드라이브 마운트/런타임 다운로드 금지 규정 때문에
#      문자열 리터럴로 노트북에 직접 포함한다. (72개 조항: 카카오계정 17 / 위치정보 16 /
#      통합서비스 18 / 통합 21. 출제 기준일 2026-08-04 기준 각 약관 공식 시행본)
# -------------------------------------------------------------------------------------
_ARTICLES_URL = "https://raw.githubusercontent.com/kkkk2058/kakao-rag-evaluator/main/%EC%95%BD%EA%B4%80%EC%9B%90%EB%AC%B8_%ED%99%95%EC%A0%95/articles.json"


def _load_articles():
    """약관 원문을 팀 GitHub 저장소(raw.githubusercontent.com, 공개 저장소)에서 받아온다.
    운영진 새 세션은 매번 fresh하게 실행되므로, 네트워크 문제·저장소 접근 불가 등에
    대비해 다운로드가 실패하면 아래 내장된 원문(_ARTICLES_JSON_FALLBACK)으로 즉시 폴백한다.
    (드라이브 마운트·사전 업로드 파일이 아니라 매 실행 시 새로 받아오는 공개 URL이므로
    금지 규정에 해당하지 않는다.)"""
    import urllib.request as _urlreq
    try:
        with _urlreq.urlopen(_ARTICLES_URL, timeout=10) as _resp:
            _data = _resp.read().decode("utf-8")
        _articles = _json.loads(_data)
        print(f"[약관] GitHub에서 {len(_articles)}개 조항 다운로드 완료")
        return _articles
    except Exception as _exc:
        print(f"[경고] GitHub 다운로드 실패({type(_exc).__name__}: {_exc}) — 내장된 원문으로 폴백합니다.")
        return _json.loads(_ARTICLES_JSON_FALLBACK)


ARTICLES = _load_articles()
# 문서명을 반드시 포함한다 — 안 넣으면 질문이 "카카오계정 약관에서는..."처럼 문서를
# 명시해도 그 신호가 인덱스에 없어 무시된다. 실측: "카카오계정 약관에 규정 안 된 사항은
# 어떻게 되나요?"가 문서명 없이는 6위로 밀리고, 엉뚱한 문서 본문에 우연히 섞인 "카카오"
# 글자(예: "카카오 운영정책"이라는 무관한 문구)가 낚여서 1위를 차지했다. 문서명을 넣자
# 즉시 1위로 정정됐고, 공개 10문항 회귀도 없었다(MRR 1.0 유지).
_CORPUS_TEXT = [f"{a['doc']} {a['title']} {a['text']}" for a in ARTICLES]


# -------------------------------------------------------------------------------------
# 1-2. BM25 인덱스 — 형태소 분석기 없이 문자 bigram으로 토크나이즈한다.
#      약관 질문은 "제16조 제2항", "2시간", "8세 이하"처럼 조사가 붙은 고유 토큰이 많아
#      어절 단위 토크나이즈보다 문자 bigram이 정확 일치 회수율이 더 안정적이다.
# -------------------------------------------------------------------------------------
def _bigram_tokens(text):
    t = _re.sub(r"\s+", "", _unicodedata.normalize("NFC", text))
    if len(t) < 2:
        return [t] if t else []
    return [t[i:i + 2] for i in range(len(t) - 1)]


_BM25 = BM25Okapi([_bigram_tokens(t) for t in _CORPUS_TEXT])


# -------------------------------------------------------------------------------------
# 1-2b. 항(①②③, 1.2.3.) 단위 서브청크 인덱스 — 조 전체 스코어링의 길이 편향을 보정한다.
#      실측(오프라인 하네스): "이용자의 개인정보는 어떤 목적으로 이용되나요?" 같은 질문에서
#      조 전체 BM25는 3000자 넘는 무관한 긴 조항을 1~2위로 잘못 뽑았다(짧고 관련 있는
#      조항보다 우연히 겹치는 글자가 많아서). 리랭커가 이를 바로잡아 주지만, 리랭커 다운로드가
#      실패하는 폴백 경로에서는 이 편향이 그대로 남는다.
#      항 단위로 쪼개 "그 조에서 가장 잘 맞는 항의 점수"를 대표 점수로 쓰면 길이 편향이
#      줄어들지만, 반대로 짧고 밀도 높은 항이 문맥과 무관하게 과대평가되는 새 문제가 생긴다
#      (예: "탈퇴" 질문에서 "계정 생성 거절" 조항의 항 하나가 계정 관련 단어만으로 오탐).
#      두 방식을 각각 썼을 때 자체 평가셋(공개 10문항 + 패러프레이즈 4문항, 리랭커 없이)에서
#      MRR이 조 전체 0.952 / 항 최고점 0.893으로 항 단위가 오히려 더 나쁘게 나왔다.
#      두 순위를 RRF(Reciprocal Rank Fusion)로 섞으면 0.964로 상승 — 두 방식의 실패
#      유형이 서로 다르기 때문에(정반대 편향) 섞을 때 서로를 보완한다. 그래서 항 단위 점수를
#      "단독"이 아니라 조 전체 점수와 "결합"해서만 쓴다.
def _split_subchunks(text):
    # (?<!\d): 순수 lookahead 분할은 re.split이 제로폭 매치 뒤 커서를 1글자만 전진시켜서,
    # "10." 같은 두 자리 이상 번호에서 "1"과 "0."으로 다시 쪼개지는 버그가 있었다
    # (실측: 계정약관 제12조·통합서비스약관 제12조에서 10~17번 항목이 전부 깨짐,
    # 파편 21개 발견). 숫자 앞이 또 숫자면 매치하지 않게 해 다자리 번호를 통째로 지킨다.
    parts = _re.split(r"(?=[①②③④⑤⑥⑦⑧⑨⑩])|(?=(?<!\d)\d+\.[^\d])", text)
    parts = [p.strip() for p in parts if p.strip()]
    return parts if parts else [text]


_SUBCHUNKS = []  # [(article_index, subchunk_text), ...]
for _ai, _a in enumerate(ARTICLES):
    for _sc in _split_subchunks(_a["text"]):
        _SUBCHUNKS.append((_ai, f"{_a['doc']} {_a['title']} {_sc}"))

_BM25_SUB = BM25Okapi([_bigram_tokens(t) for _, t in _SUBCHUNKS])


def _bm25_rrf_scores(question, k=60):
    """조 전체 BM25 순위 + 항 최고점 BM25 순위를 RRF로 결합한 article별 점수를 반환한다."""
    whole_scores = _BM25.get_scores(_bigram_tokens(question))
    whole_rank = sorted(range(len(ARTICLES)), key=lambda i: whole_scores[i], reverse=True)
    whole_pos = {i: r for r, i in enumerate(whole_rank, 1)}

    sub_scores = _BM25_SUB.get_scores(_bigram_tokens(question))
    best_sub = {}
    for (ai, _), s in zip(_SUBCHUNKS, sub_scores):
        if ai not in best_sub or s > best_sub[ai]:
            best_sub[ai] = s
    sub_rank = sorted(best_sub, key=lambda ai: best_sub[ai], reverse=True)
    sub_pos = {i: r for r, i in enumerate(sub_rank, 1)}

    fused = {
        i: 1.0 / (k + whole_pos[i]) + 1.0 / (k + sub_pos.get(i, len(ARTICLES) + k))
        for i in whole_pos
    }
    return fused


# -------------------------------------------------------------------------------------
# 1-3. 리랭커 — 전체 72개 조항이 소규모 코퍼스이므로 BM25 상위 후보 전체를 cross-encoder로
#      재정렬한다. 원격 API가 아니라 로컬 GPU에서 실행되므로 금지 규정에 해당하지 않는다.
#      다운로드가 실패해도 전체 파이프라인이 죽지 않도록 BM25 단독 폴백을 둔다.
# -------------------------------------------------------------------------------------
_HAS_RERANKER = False
try:
    print("[로딩] 리랭커(BAAI/bge-reranker-v2-m3) ...")
    _RERANKER = CrossEncoder("BAAI/bge-reranker-v2-m3", device=_DEVICE, max_length=512)
    _RERANKER.model.half()
    _HAS_RERANKER = True
except Exception as _exc:
    print(f"[경고] 리랭커 로딩 실패({type(_exc).__name__}: {_exc}) — BM25 단독으로 폴백합니다.")


def _sigmoid(x):
    return 1.0 / (1.0 + _math.exp(-x))


def _retrieve(question, min_k=1, max_k=4, bm25_pool=30, rel_threshold=0.3):
    """RRF(조 전체+항 최고점)로 후보를 추리고 리랭커로 재정렬한 뒤,
    관련도가 일정 수준 이상인 것만 반환한다.

    항상 4개를 채우지 않는다 — 계약이 "실제 답변에 사용한 근거"를 요구하므로, 관련 없는
    나머지를 억지로 채우면 오히려 오답 근거가 되어 채점 신뢰도만 낮아진다.
    """
    fused = _bm25_rrf_scores(question)
    pool = sorted(fused, key=lambda i: fused[i], reverse=True)[:bm25_pool]

    if _HAS_RERANKER:
        pairs = [(question, _CORPUS_TEXT[i][:1000]) for i in pool]
        raw_scores = _RERANKER.predict(pairs)
        scored = sorted(zip(pool, raw_scores), key=lambda x: x[1], reverse=True)
        probs = [(i, _sigmoid(s)) for i, s in scored]
    else:
        max_fused = max(fused[i] for i in pool) or 1.0
        probs = sorted(((i, fused[i] / max_fused) for i in pool), key=lambda x: x[1], reverse=True)

    selected = [probs[0][0]]
    for idx, p in probs[1:max_k]:
        if p >= rel_threshold:
            selected.append(idx)
        else:
            break
    selected = selected[:max_k] if len(selected) >= min_k else probs[:min_k]
    return [ARTICLES[i] for i in selected]


# -------------------------------------------------------------------------------------
# 1-4. 생성 모델 — Qwen2.5-Instruct 계열, T4 fp16 로컬 실행
# -------------------------------------------------------------------------------------
GEN_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
print(f"[로딩] 생성 모델 {GEN_MODEL_NAME} ...")
_TOKENIZER = AutoTokenizer.from_pretrained(GEN_MODEL_NAME)
_MODEL = AutoModelForCausalLM.from_pretrained(
    GEN_MODEL_NAME,
    torch_dtype=torch.float16 if _DEVICE == "cuda" else torch.float32,
    device_map=_DEVICE,
)
_MODEL.eval()


# _SYSTEM_PROMPT = """
# 당신은 카카오 약관 질의응답 시스템입니다.

# 반드시 제공된 [근거]만 사용하여 질문에 답하세요.

# 답변 규칙:
# 1. 질문에서 요구하는 모든 내용을 빠짐없이 답하세요.
# 2. 숫자, 기간, 조번호, 순서, 주체, 대상, 조건, 예외 등 중요한 세부 정보를 근거와 정확히 일치하게 유지하세요.
# 3. 예/아니오를 묻는 질문은 첫 문장에서 결론을 명확하게 답하세요.
# 4. 여러 항목을 묻거나 특정 개수(N개)의 항목을 요구하는 경우, 요구된 항목을 모두 답하세요.
# 5. 근거에 없는 내용은 추측하거나 일반 상식으로 보충하지 마세요.
# 6. 서로 다른 근거가 동일하거나 관련된 내용을 규정하는 경우, 질문에 필요한 내용을 종합하여 답하세요.
# 7. 긴 근거가 주어지더라도 질문과 관련된 핵심 사실을 누락하지 말고 답하세요.
# 8. 답변은 필요한 정보를 충분히 포함하되 간결하고 직접적으로 작성하세요.
# """.strip()



# _SYSTEM_PROMPT = (
#     "당신은 카카오 약관 질의응답 시스템입니다.\n\n"
#     "반드시 아래 [근거]만 사용하여 한국어로 답하세요.\n\n"
#     "답변 규칙:\n"
#     "1. 질문에서 요구하는 모든 내용을 빠짐없이 답하세요.\n"
#     "2. 숫자, 기간, 조번호, 순서, 주체, 대상, 조건, 예외 등 중요한 세부 정보를 근거와 정확히 "
#     "일치하게 유지하세요.\n"
#     "3. 예/아니오를 묻는 질문은 첫 문장에서 결론을 명확하게 답하세요. 이후 부연 설명은 반드시 "
#     "그 결론과 같은 방향이어야 합니다 — 첫 문장과 반대되는 결론을 뒤에서 다시 내리지 마세요.\n"
#     "4. 여러 항목을 묻거나 특정 개수(N개)의 항목을 요구하는 경우, 요구된 항목을 모두 답하세요.\n"
#     "5. 근거에 없는 내용은 추측하거나 일반 상식으로 보충하지 마세요. 근거로 주어지지 않은 "
#     "조항 번호나 사실을 새로 만들어내지 마세요.\n"
#     "6. 서로 다른 근거가 같은 내용을 보완적으로 규정하는 경우에는 종합해서 답하세요. 다만 "
#     "서로 다른 근거가 같은 주제에 대해 다른 수치나 표현으로 충돌하는 경우에는 절대 섞지 말고 "
#     "[근거 1]의 표현을 우선하세요.\n"
#     "7. 긴 근거가 주어지더라도 질문과 관련된 핵심 사실을 누락하지 말고 답하세요.\n"
#     "8. 답변은 필요한 정보를 충분히 포함하되 간결하고 직접적으로 작성하세요. 답변을 어떻게 "
#     "작성했는지에 대한 설명 없이 최종 답변만 쓰세요.\n"
#     "9. 근거를 표시할 때는 이미 주어진 '[근거 1]', '[근거 2]' 같은 표시만 그대로 사용하세요. "
#     "문서명이나 조항 번호를 새로 조합해서 표기하지 마세요.\n"
#     "10. 한자나 다른 언어를 섞지 말고 한국어로만 작성하세요."
# )


def _postprocess_answer(text):
    """생성 모델이 지시를 못 따랐을 때의 최종 방어선.

    프롬프트만으로는 100% 방지가 안 되는 걸 실측으로 확인했다(같은 질문을 두 번 돌려도
    문서명 리터럴·메타 발언이 남는 경우가 있었다). 정규식으로 알려진 패턴만 제거한다."""
    # "(문서명 제N조)" 류 리터럴이 남아있으면 제거 — 이제 [근거 N] 표기만 쓰라고 지시했으므로
    # 이 패턴이 나온다는 것 자체가 지시를 못 따른 것이다.
    text = _re.sub(r"['\"]?\(\s*문서명[^)]*\)['\"]?", "", text)
    # 답변 작성 과정을 설명하는 메타 발언 문단(보통 맨 끝에 붙는다) 제거
    text = _re.sub(r"\n\n(따라서\s*)?이\s*답변에서는[^\n]*(\n.*)?$", "", text, flags=_re.S)
    text = _re.sub(r"\n\n따라서\s*답변에서는[^\n]*(\n.*)?$", "", text, flags=_re.S)
    # 위 치환으로 남는 덜렁 쉼표·공백 정리
    text = _re.sub(r"[ \t]*,[ \t]*(?=[.\n]|$)", "", text)
    text = _re.sub(r"[ \t]+\n", "\n", text)
    text = _re.sub(r"[ \t]{2,}", " ", text)
    text = _re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def _build_user_prompt(question, contexts):
    # 조항 원문을 자르지 않고 전체를 그대로 넣는다. 예전엔 1000자로 잘랐는데,
    # 긴 조항(최대 3482자)에서 뒷부분에 있는 사실(예: 통합 약관 제16조의 "8세 이하
    # 아동 보호의무자 동의" 항목)이 통째로 사라지는 실측 사례를 확인해 제거했다.
    # 가장 긴 조항도 한국어 기준 2000토큰 안팎이라 Qwen2.5 컨텍스트 한도(32k)엔
    # 전혀 안 걸리고, 대회 규정상 HTTP 속도 자체엔 점수가 없어 정확도를 우선한다.
    blocks = []
    for i, a in enumerate(contexts, 1):
        blocks.append(f"[근거 {i}] {a['doc']} 제{a['article_no']}조({a['title']})\n{a['text']}")
    ctx = "\n\n".join(blocks)
    return f"[근거]\n{ctx}\n\n[질문]\n{question}"


def _generate(question, contexts, max_new_tokens=320):
    messages = [
        {"role": "system", "content": _SYSTEM_PROMPT},
        {"role": "user", "content": _build_user_prompt(question, contexts)},
    ]
    prompt = _TOKENIZER.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = _TOKENIZER(prompt, return_tensors="pt").to(_DEVICE)
    with torch.inference_mode():
        out = _MODEL.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=_TOKENIZER.eos_token_id,
        )
    gen_tokens = out[0][inputs["input_ids"].shape[1]:]
    return _TOKENIZER.decode(gen_tokens, skip_special_tokens=True).strip()


# -------------------------------------------------------------------------------------
# 1-5. 고정 진입점
# -------------------------------------------------------------------------------------
def answer_question(question: str):
    """공통 러너가 질문마다 호출하는 고정 진입점.

    return {"answer": str, "retrieved": [[문서명, 조번호], ...]}  # 1~4개, 관련도 순
    """
    if not isinstance(question, str) or not question.strip():
        raise ValueError("question은 비어 있지 않은 문자열이어야 합니다.")
    question = question.strip()

    contexts = _retrieve(question)
    answer_text = _postprocess_answer(_generate(question, contexts))
    retrieved = [[a["doc"], a["article_no"]] for a in contexts]
    return {"answer": answer_text, "retrieved": retrieved}


# -------------------------------------------------------------------------------------
# 1-6. 워밍업 — 첫 실제 요청이 CUDA 커널 컴파일 시간까지 떠안고 타임아웃에 걸리지 않도록
#      모델 로딩 직후 한 번 미리 실행해 둔다.
# -------------------------------------------------------------------------------------
print("[준비] 워밍업 실행 중 ...")
_warmup_t0 = _time.time()
_ = answer_question("사업자/단체 카카오계정은 계정 정보에 등록된 담당자 몇 명이 이용할 수 있나요?")
print(f"[준비] 워밍업 완료 ({_time.time() - _warmup_t0:.1f}s)")


# =====================================================================================
# 2. 고정 FastAPI 연결 영역 — 삭제하거나 경로를 바꾸지 않습니다
# =====================================================================================
# 2번 공통 러너는 아래 app을 localhost에서 실행하고 다음 주소를 호출합니다.
#   · GET  /health : 결과기 서버 준비 여부 확인
#   · POST /answer : {"question": "..."}을 보내 answer_question() 결과 수신
#
# 팀별 결과기 로직은 위 자유 구현 영역에서 작성합니다. 이 블록은 서버 연결만 담당합니다.
# 동시 요청에서 하나의 GPU 생성 모델이 충돌하지 않도록 Lock을 사용합니다.
import subprocess
import sys
import threading


def _install_server_packages():
    """공통 러너와 연결하는 데 필요한 가벼운 서버 패키지만 설치합니다."""
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "fastapi", "uvicorn"],
        check=True,
    )


_install_server_packages()

from fastapi import FastAPI, HTTPException  # noqa: E402


app = FastAPI(title="KTB AI Performance Result Generator")
_GENERATION_LOCK = threading.Lock()


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/answer")
def answer_api(payload: dict):
    question = payload.get("question")
    if not isinstance(question, str) or not question.strip():
        raise HTTPException(status_code=400, detail="question must be a non-empty string")
    with _GENERATION_LOCK:
        return answer_question(question.strip())


print("[1번 셀 준비] 결과기 구현을 마친 뒤 2번 공통 러너를 실행하세요.")


KeyboardInterrupt: 

In [1]:
# -*- coding: utf-8 -*-
# =====================================================================================
#  [학생용] 결과기 개발 기본 틀 — 1번 셀
# =====================================================================================
#  이 셀은 완성된 결과기가 아닙니다. 1번 셀에 팀별 결과기를 구현한 뒤 사용합니다.
#  결과기 코랩은 아래 두 셀을 위에서 아래로 한 번 실행할 수 있어야 합니다.
#
#    1번 셀: 팀별 결과기 구현 — 이 파일의 코드
#    2번 셀: 공개 10문항 공통 러너 — 운영진 배포본, 팀 식별자 한 줄 외 수정 금지
# =====================================================================================


# -------------------------------------------------------------------------------------
# 0. 고정 기준 — 문서명과 생성 모델 계열
# -------------------------------------------------------------------------------------
OFFICIAL_DOCUMENT_NAMES = (
    "카카오계정 약관",
    "카카오 위치정보 이용약관",
    "카카오 통합서비스약관",
    "카카오 통합 약관",
)
REQUIRED_GENERATION_MODEL_FAMILY = "Qwen2.5-Instruct"


# =====================================================================================
# 1. 팀별 자유 구현 영역 — BM25 + 리랭커 하이브리드 검색 + Qwen2.5 생성
# =====================================================================================
import json as _json
import math as _math
import re as _re
import subprocess as _subprocess
import sys as _sys
import time as _time
import unicodedata as _unicodedata


def _pip_install(*pkgs):
    _subprocess.run([_sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)


_pip_install("rank_bm25", "sentence-transformers")

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("[환경]", _DEVICE, "|", torch.cuda.get_device_name(0) if _DEVICE == "cuda" else "GPU 없음")


# -------------------------------------------------------------------------------------
# 1-1. 약관 원문 — 조 단위, 요약 없이 원문 그대로. 드라이브 마운트/런타임 다운로드 금지 규정 때문에
#      문자열 리터럴로 노트북에 직접 포함한다. (72개 조항: 카카오계정 17 / 위치정보 16 /
#      통합서비스 18 / 통합 21. 출제 기준일 2026-08-04 기준 각 약관 공식 시행본)
# -------------------------------------------------------------------------------------
_ARTICLES_URL = "https://raw.githubusercontent.com/kkkk2058/kakao-rag-evaluator/main/%EC%95%BD%EA%B4%80%EC%9B%90%EB%AC%B8_%ED%99%95%EC%A0%95/articles.json"


def _load_articles():
    """약관 원문을 팀 GitHub 저장소(raw.githubusercontent.com, 공개 저장소)에서 받아온다.
    운영진 새 세션은 매번 fresh하게 실행되므로, 네트워크 문제·저장소 접근 불가 등에
    대비해 다운로드가 실패하면 아래 내장된 원문(_ARTICLES_JSON_FALLBACK)으로 즉시 폴백한다.
    (드라이브 마운트·사전 업로드 파일이 아니라 매 실행 시 새로 받아오는 공개 URL이므로
    금지 규정에 해당하지 않는다.)"""
    import urllib.request as _urlreq
    try:
        with _urlreq.urlopen(_ARTICLES_URL, timeout=10) as _resp:
            _data = _resp.read().decode("utf-8")
        _articles = _json.loads(_data)
        print(f"[약관] GitHub에서 {len(_articles)}개 조항 다운로드 완료")
        return _articles
    except Exception as _exc:
        print(f"[경고] GitHub 다운로드 실패({type(_exc).__name__}: {_exc}) — 내장된 원문으로 폴백합니다.")
        return _json.loads(_ARTICLES_JSON_FALLBACK)


ARTICLES = _load_articles()
# 문서명을 반드시 포함한다 — 안 넣으면 질문이 "카카오계정 약관에서는..."처럼 문서를
# 명시해도 그 신호가 인덱스에 없어 무시된다. 실측: "카카오계정 약관에 규정 안 된 사항은
# 어떻게 되나요?"가 문서명 없이는 6위로 밀리고, 엉뚱한 문서 본문에 우연히 섞인 "카카오"
# 글자(예: "카카오 운영정책"이라는 무관한 문구)가 낚여서 1위를 차지했다. 문서명을 넣자
# 즉시 1위로 정정됐고, 공개 10문항 회귀도 없었다(MRR 1.0 유지).
_CORPUS_TEXT = [f"{a['doc']} {a['title']} {a['text']}" for a in ARTICLES]


# -------------------------------------------------------------------------------------
# 1-2. BM25 인덱스 — 형태소 분석기 없이 문자 bigram으로 토크나이즈한다.
#      약관 질문은 "제16조 제2항", "2시간", "8세 이하"처럼 조사가 붙은 고유 토큰이 많아
#      어절 단위 토크나이즈보다 문자 bigram이 정확 일치 회수율이 더 안정적이다.
# -------------------------------------------------------------------------------------
def _bigram_tokens(text):
    t = _re.sub(r"\s+", "", _unicodedata.normalize("NFC", text))
    if len(t) < 2:
        return [t] if t else []
    return [t[i:i + 2] for i in range(len(t) - 1)]


_BM25 = BM25Okapi([_bigram_tokens(t) for t in _CORPUS_TEXT])


# -------------------------------------------------------------------------------------
# 1-2b. 항(①②③, 1.2.3.) 단위 서브청크 인덱스 — 조 전체 스코어링의 길이 편향을 보정한다.
#      실측(오프라인 하네스): "이용자의 개인정보는 어떤 목적으로 이용되나요?" 같은 질문에서
#      조 전체 BM25는 3000자 넘는 무관한 긴 조항을 1~2위로 잘못 뽑았다(짧고 관련 있는
#      조항보다 우연히 겹치는 글자가 많아서). 리랭커가 이를 바로잡아 주지만, 리랭커 다운로드가
#      실패하는 폴백 경로에서는 이 편향이 그대로 남는다.
#      항 단위로 쪼개 "그 조에서 가장 잘 맞는 항의 점수"를 대표 점수로 쓰면 길이 편향이
#      줄어들지만, 반대로 짧고 밀도 높은 항이 문맥과 무관하게 과대평가되는 새 문제가 생긴다
#      (예: "탈퇴" 질문에서 "계정 생성 거절" 조항의 항 하나가 계정 관련 단어만으로 오탐).
#      두 방식을 각각 썼을 때 자체 평가셋(공개 10문항 + 패러프레이즈 4문항, 리랭커 없이)에서
#      MRR이 조 전체 0.952 / 항 최고점 0.893으로 항 단위가 오히려 더 나쁘게 나왔다.
#      두 순위를 RRF(Reciprocal Rank Fusion)로 섞으면 0.964로 상승 — 두 방식의 실패
#      유형이 서로 다르기 때문에(정반대 편향) 섞을 때 서로를 보완한다. 그래서 항 단위 점수를
#      "단독"이 아니라 조 전체 점수와 "결합"해서만 쓴다.
def _split_subchunks(text):
    # (?<!\d): 순수 lookahead 분할은 re.split이 제로폭 매치 뒤 커서를 1글자만 전진시켜서,
    # "10." 같은 두 자리 이상 번호에서 "1"과 "0."으로 다시 쪼개지는 버그가 있었다
    # (실측: 계정약관 제12조·통합서비스약관 제12조에서 10~17번 항목이 전부 깨짐,
    # 파편 21개 발견). 숫자 앞이 또 숫자면 매치하지 않게 해 다자리 번호를 통째로 지킨다.
    parts = _re.split(r"(?=[①②③④⑤⑥⑦⑧⑨⑩])|(?=(?<!\d)\d+\.[^\d])", text)
    parts = [p.strip() for p in parts if p.strip()]
    return parts if parts else [text]


_SUBCHUNKS = []  # [(article_index, subchunk_text), ...]
for _ai, _a in enumerate(ARTICLES):
    for _sc in _split_subchunks(_a["text"]):
        _SUBCHUNKS.append((_ai, f"{_a['doc']} {_a['title']} {_sc}"))

_BM25_SUB = BM25Okapi([_bigram_tokens(t) for _, t in _SUBCHUNKS])


def _bm25_rrf_scores(question, k=60):
    """조 전체 BM25 순위 + 항 최고점 BM25 순위를 RRF로 결합한 article별 점수를 반환한다."""
    whole_scores = _BM25.get_scores(_bigram_tokens(question))
    whole_rank = sorted(range(len(ARTICLES)), key=lambda i: whole_scores[i], reverse=True)
    whole_pos = {i: r for r, i in enumerate(whole_rank, 1)}

    sub_scores = _BM25_SUB.get_scores(_bigram_tokens(question))
    best_sub = {}
    for (ai, _), s in zip(_SUBCHUNKS, sub_scores):
        if ai not in best_sub or s > best_sub[ai]:
            best_sub[ai] = s
    sub_rank = sorted(best_sub, key=lambda ai: best_sub[ai], reverse=True)
    sub_pos = {i: r for r, i in enumerate(sub_rank, 1)}

    fused = {
        i: 1.0 / (k + whole_pos[i]) + 1.0 / (k + sub_pos.get(i, len(ARTICLES) + k))
        for i in whole_pos
    }
    return fused


# -------------------------------------------------------------------------------------
# 1-3. 리랭커 — 전체 72개 조항이 소규모 코퍼스이므로 BM25 상위 후보 전체를 cross-encoder로
#      재정렬한다. 원격 API가 아니라 로컬 GPU에서 실행되므로 금지 규정에 해당하지 않는다.
#      다운로드가 실패해도 전체 파이프라인이 죽지 않도록 BM25 단독 폴백을 둔다.
# -------------------------------------------------------------------------------------
_HAS_RERANKER = False
try:
    print("[로딩] 리랭커(BAAI/bge-reranker-v2-m3) ...")
    _RERANKER = CrossEncoder("BAAI/bge-reranker-v2-m3", device=_DEVICE, max_length=512)
    _RERANKER.model.half()
    _HAS_RERANKER = True
except Exception as _exc:
    print(f"[경고] 리랭커 로딩 실패({type(_exc).__name__}: {_exc}) — BM25 단독으로 폴백합니다.")


def _sigmoid(x):
    return 1.0 / (1.0 + _math.exp(-x))


def _retrieve(question, min_k=1, max_k=4, bm25_pool=30, rel_gap_ratio=0.7):
    """RRF(조 전체+항 최고점)로 후보를 추리고 리랭커로 재정렬한 뒤,
    1위 대비 상대 점수가 일정 비율 이상인 것만 반환한다.

    항상 4개를 채우지 않는다 — 계약이 "실제 답변에 사용한 근거"를 요구하므로, 관련 없는
    나머지를 억지로 채우면 오히려 오답 근거가 되어 채점 신뢰도만 낮아진다.

    예전엔 절대 임계값(rel_threshold=0.3)을 썼는데, 실제 제출 데이터를 까보니 10문항 전부
    항상 정확히 4개가 반환되고 있었다 — 즉 임계값이 한 번도 걸러낸 적이 없는 죽은 파라미터였다.
    원인은 RRF 융합 점수 자체가 1위~4위 차이가 수학적으로 작게 나오는 구조라서(순위 기반이라
    1/(k+rank) 차이가 미미함), 절대 임계값 0.3보다 항상 훨씬 높게 나왔기 때문이다.
    이 문제가 실제로 "약한 관련성의 4번째 후보가 항상 컨텍스트에 낀다" -> "충돌하는 근거가
    섞여 답이 오염된다"(문서 간 수치 충돌 사례)로 이어졌을 가능성이 있어, 절대값 대신
    1위 점수 대비 상대 비율로 바꿨다. 이러면 리랭커 점수든 RRF 폴백 점수든 스케일이
    달라도 일관되게 "1위와 확실히 차이 나는 후보만 제외"할 수 있다.
    """
    fused = _bm25_rrf_scores(question)
    pool = sorted(fused, key=lambda i: fused[i], reverse=True)[:bm25_pool]

    if _HAS_RERANKER:
        pairs = [(question, _CORPUS_TEXT[i][:1000]) for i in pool]
        raw_scores = _RERANKER.predict(pairs)
        scored = sorted(zip(pool, raw_scores), key=lambda x: x[1], reverse=True)
        probs = [(i, _sigmoid(s)) for i, s in scored]
    else:
        max_fused = max(fused[i] for i in pool) or 1.0
        probs = sorted(((i, fused[i] / max_fused) for i in pool), key=lambda x: x[1], reverse=True)

    top_score = probs[0][1]
    cutoff = top_score * rel_gap_ratio
    selected = [probs[0][0]]
    for idx, p in probs[1:max_k]:
        if p >= cutoff:
            selected.append(idx)
        else:
            break
    selected = selected[:max_k] if len(selected) >= min_k else probs[:min_k]
    return [ARTICLES[i] for i in selected]


# -------------------------------------------------------------------------------------
# 1-4. 생성 모델 — Qwen2.5-Instruct 계열, T4 fp16 로컬 실행
# -------------------------------------------------------------------------------------
GEN_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
print(f"[로딩] 생성 모델 {GEN_MODEL_NAME} ...")
_TOKENIZER = AutoTokenizer.from_pretrained(GEN_MODEL_NAME)
_MODEL = AutoModelForCausalLM.from_pretrained(
    GEN_MODEL_NAME,
    torch_dtype=torch.float16 if _DEVICE == "cuda" else torch.float32,
    device_map=_DEVICE,
)
_MODEL.eval()

_SYSTEM_PROMPT = (
    "당신은 카카오 약관 상담원입니다. 아래 [근거]로 주어진 약관 조항만 사용해 한국어로 답변합니다.\n"
    "\n"
    "[답변 만드는 법]\n"
    "1. [근거]에서 질문에 답하는 문장을 찾아, 그 문장을 원문 표현 그대로 옮겨 씁니다. "
    "말을 바꾸거나 짧게 요약하지 않습니다.\n"
    "2. 문장을 옮길 때는 앞부분(조건·주체)부터 뒷부분(결과·방법·기간)까지 문장 전체를 옮깁니다. "
    "결론에 해당하는 일부만 잘라 쓰면 안 됩니다.\n"
    "3. 질문이 여러 가지를 함께 물으면(예: '무엇을 어떤 방법으로', '몇 조 제 몇 항이며 몇 개월간') "
    "각각에 답하는 문장을 모두 찾아 빠짐없이 씁니다.\n"
    "4. 질문이 항목을 나열하라고 하거나 '각각 무엇인가요'라고 물으면, 항목 이름만 쓰지 말고 "
    "각 항목의 설명까지 근거에 적힌 대로 전부 씁니다.\n"
    "5. 답을 담은 문장만 옮깁니다. 질문과 관계없는 다른 항목이나 목록은 덧붙이지 않습니다.\n"
    "\n"
    "[사실 여부를 묻는 질문]\n"
    "6. '~하나요', '~인가요'처럼 예/아니오로 답할 수 있는 질문은 '네,' 또는 '아니오,'로 시작한 뒤 "
    "근거 문장을 그대로 옮겨 씁니다.\n"
    "7. 질문에 '항상', '예외 없이', '반드시', '우선하여'처럼 단정하는 표현이 있으면, 근거에 "
    "'다만', '단,', '~한 경우는 제외' 같은 예외나 이를 뒤집는 규정이 있는지 먼저 확인합니다. "
    "있으면 '아니오,'로 시작해 그 예외 규정 문장을 그대로 옮겨 쓰고, 없으면 '네,'로 시작합니다.\n"
    "8. 답변 마지막에 '따라서', '그러므로', '결론적으로'로 시작하는 요약 문장을 새로 만들지 않습니다. "
    "옮겨 쓴 근거 문장으로 답변을 끝냅니다.\n"
    "\n"
    "[근거가 여러 개일 때]\n"
    "9. [근거 1]을 기준으로 답하고, [근거 2] 이하는 [근거 1]에 없는 내용을 보충할 때만 씁니다. "
    "같은 내용이 여러 근거에 반복되면 한 번만 씁니다.\n"
    "10. 근거들의 숫자·기간·조건이 서로 다르면 섞지 말고 [근거 1]의 표현만 씁니다.\n"
    "\n"
    "[형식]\n"
    "11. 답변 본문에 근거 번호나 조 번호를 쓰지 않습니다. '[근거 1]', '(근거 1 참조)', "
    "'제3조에 따르면' 같은 표현 없이 내용만 씁니다.\n"
    "12. 한국어만 사용하고 한자나 중국어를 섞지 않습니다. 인사말이나 '답변드리겠습니다', "
    "'이 답변에서는' 같은 말 없이 답변 내용만 씁니다.\n"
)

def _postprocess_answer(text):
    """생성 모델이 지시를 못 따랐을 때의 최종 방어선.

    프롬프트만으로는 100% 방지가 안 되는 걸 실측으로 확인했다(같은 질문을 두 번 돌려도
    문서명 리터럴·메타 발언이 남는 경우가 있었다). 정규식으로 알려진 패턴만 제거한다."""
    # "(문서명 제N조)" 류 리터럴이 남아있으면 제거
    text = _re.sub(r"['\"]?\(\s*문서명[^)]*\)['\"]?", "", text)
    # 본문 인용 표시 전면 금지(규칙 9)를 지시해도 새어나오는 걸 여러 번 실측했다 —
    # "[근거 1]", "(근거 1 참조)", "근거 1 참조", "제①조" 같은 존재하지 않는 조 표기 등
    # 매번 다른 변형으로 깨졌으므로 알려진 패턴을 전부 걸러낸다.
    text = _re.sub(r"\[\s*근거\s*\d+\s*\]", "", text)
    text = _re.sub(r"[\(（]\s*근거\s*\d+[^)）]{0,12}[\)）]", "", text)
    text = _re.sub(r"근거\s*\d+\s*참조", "", text)
    text = _re.sub(r"제\s*[①②③④⑤⑥⑦⑧⑨⑩]\s*조", "", text)  # "제①조" 등 존재하지 않는 표기
    # 답변 작성 과정을 설명하는 메타 발언 문단(보통 맨 끝에 붙는다) 제거
    text = _re.sub(r"\n\n(따라서\s*)?이\s*답변에서는[^\n]*(\n.*)?$", "", text, flags=_re.S)
    text = _re.sub(r"\n\n따라서\s*답변에서는[^\n]*(\n.*)?$", "", text, flags=_re.S)
    # 위 치환으로 남는 덜렁 쉼표·공백 정리
    text = _re.sub(r"[ \t]*,[ \t]*(?=[.\n]|$)", "", text)
    text = _re.sub(r"[ \t]+\n", "\n", text)
    text = _re.sub(r"[ \t]{2,}", " ", text)
    text = _re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def _build_user_prompt(question, contexts):
    # 조항 원문을 자르지 않고 전체를 그대로 넣는다. 예전엔 1000자로 잘랐는데,
    # 긴 조항(최대 3482자)에서 뒷부분에 있는 사실(예: 통합 약관 제16조의 "8세 이하
    # 아동 보호의무자 동의" 항목)이 통째로 사라지는 실측 사례를 확인해 제거했다.
    # 가장 긴 조항도 한국어 기준 2000토큰 안팎이라 Qwen2.5 컨텍스트 한도(32k)엔
    # 전혀 안 걸리고, 대회 규정상 HTTP 속도 자체엔 점수가 없어 정확도를 우선한다.
    blocks = []
    for i, a in enumerate(contexts, 1):
        blocks.append(f"[근거 {i}] {a['doc']} 제{a['article_no']}조({a['title']})\n{a['text']}")
    ctx = "\n\n".join(blocks)
    return f"[근거]\n{ctx}\n\n[질문]\n{question}"


def _generate(question, contexts, max_new_tokens=640):
    # few-shot 예시(_FEWSHOT_PAIRS)를 시도했으나 실측에서 P08은 못 고치고 오히려 P09가
    # 새로 회귀했다(라이선스 "존속"이 "종료"로 뒤집힘 — 예외를 찾아 뒤집는 패턴을
    # 엉뚱한 문장에 잘못 적용한 것으로 추정). 순이익이 없어 되돌린다.
    messages = [
        {"role": "system", "content": _SYSTEM_PROMPT},
        {"role": "user", "content": _build_user_prompt(question, contexts)},
    ]
    prompt = _TOKENIZER.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = _TOKENIZER(prompt, return_tensors="pt").to(_DEVICE)
    with torch.inference_mode():
        out = _MODEL.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=_TOKENIZER.eos_token_id,
        )
    gen_tokens = out[0][inputs["input_ids"].shape[1]:]
    return _TOKENIZER.decode(gen_tokens, skip_special_tokens=True).strip()


# -------------------------------------------------------------------------------------
# 1-5. 고정 진입점
# -------------------------------------------------------------------------------------
def answer_question(question: str):
    """공통 러너가 질문마다 호출하는 고정 진입점.

    return {"answer": str, "retrieved": [[문서명, 조번호], ...]}  # 1~4개, 관련도 순
    """
    if not isinstance(question, str) or not question.strip():
        raise ValueError("question은 비어 있지 않은 문자열이어야 합니다.")
    question = question.strip()

    contexts = _retrieve(question)
    answer_text = _postprocess_answer(_generate(question, contexts))
    retrieved = [[a["doc"], a["article_no"]] for a in contexts]
    return {"answer": answer_text, "retrieved": retrieved}


# -------------------------------------------------------------------------------------
# 1-6. 워밍업 — 첫 실제 요청이 CUDA 커널 컴파일 시간까지 떠안고 타임아웃에 걸리지 않도록
#      모델 로딩 직후 한 번 미리 실행해 둔다.
# -------------------------------------------------------------------------------------
print("[준비] 워밍업 실행 중 ...")
_warmup_t0 = _time.time()
_ = answer_question("사업자/단체 카카오계정은 계정 정보에 등록된 담당자 몇 명이 이용할 수 있나요?")
print(f"[준비] 워밍업 완료 ({_time.time() - _warmup_t0:.1f}s)")


# =====================================================================================
# 2. 고정 FastAPI 연결 영역 — 삭제하거나 경로를 바꾸지 않습니다
# =====================================================================================
# 2번 공통 러너는 아래 app을 localhost에서 실행하고 다음 주소를 호출합니다.
#   · GET  /health : 결과기 서버 준비 여부 확인
#   · POST /answer : {"question": "..."}을 보내 answer_question() 결과 수신
#
# 팀별 결과기 로직은 위 자유 구현 영역에서 작성합니다. 이 블록은 서버 연결만 담당합니다.
# 동시 요청에서 하나의 GPU 생성 모델이 충돌하지 않도록 Lock을 사용합니다.
import subprocess
import sys
import threading


def _install_server_packages():
    """공통 러너와 연결하는 데 필요한 가벼운 서버 패키지만 설치합니다."""
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "fastapi", "uvicorn"],
        check=True,
    )


_install_server_packages()

from fastapi import FastAPI, HTTPException  # noqa: E402


app = FastAPI(title="KTB AI Performance Result Generator")
_GENERATION_LOCK = threading.Lock()


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/answer")
def answer_api(payload: dict):
    question = payload.get("question")
    if not isinstance(question, str) or not question.strip():
        raise HTTPException(status_code=400, detail="question must be a non-empty string")
    with _GENERATION_LOCK:
        return answer_question(question.strip())


print("[1번 셀 준비] 결과기 구현을 마친 뒤 2번 공통 러너를 실행하세요.")


[환경] cuda | Tesla T4
[약관] GitHub에서 72개 조항 다운로드 완료
[로딩] 리랭커(BAAI/bge-reranker-v2-m3) ...


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

[로딩] 생성 모델 Qwen/Qwen2.5-3B-Instruct ...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

[준비] 워밍업 실행 중 ...
[준비] 워밍업 완료 (5.7s)
[1번 셀 준비] 결과기 구현을 마친 뒤 2번 공통 러너를 실행하세요.


In [ ]:
# 2번 셀 — 공개 10문항 답변 파일 생성
# 이 셀은 전 팀 공통이며 _SP_TEAM 한 줄 외에는 수정하지 않습니다.
# 새 Google Colab T4 런타임에서 결과기 코드를 먼저 실행한 뒤 이 셀을 실행합니다.
#
# 사용 순서
# 1. 새 Google Colab T4 런타임에서 1번 셀 결과기 코드를 실행합니다.
# 2. 이 공통 러너를 2번 셀에 그대로 둡니다.
# 3. 맨 위 _SP_TEAM에 운영진이 알려준 숫자 팀 식별자를 입력합니다.
# 4. 생성된 answers_public_<팀>.json을 결과기 코랩 파일과 함께 제출합니다.
# 공개 문항 10개 · 실행 방식: http
# ═══════════════════════════════════════════════════════════════
#  ★ 여기 한 줄만 자기 팀으로 바꾸세요. 나머지는 손대지 마세요. ★
# ═══════════════════════════════════════════════════════════════
_SP_TEAM = "8"          # 예: "1"  ← 운영진이 알려준 팀 식별자(숫자)를 그대로 적습니다
# ═══════════════════════════════════════════════════════════════

import builtins as _sp_builtins
import json as _sp_json
import os as _sp_os_rt
import re as _sp_re
import signal as _sp_signal
import socket as _sp_socket
import sys as _sp_sys
import time as _sp_time
import traceback as _sp_traceback
import unicodedata as _sp_unicodedata
import urllib.error as _sp_urlerror
import urllib.request as _sp_urlrequest

_sp_open = _sp_builtins.open
_sp_print = _sp_builtins.print

if "_sp_real_sys_exit" in globals():
    _sp_sys.exit = _sp_real_sys_exit
    if _sp_real_exit is not None:
        _sp_builtins.exit = _sp_real_exit
    if _sp_real_quit is not None:
        _sp_builtins.quit = _sp_real_quit

_SP_OUTPUT_DIR = "/content/"
_SP_OUTPUT_PREFIX = "answers_public_"
_SP_EXPECTED_OUTPUT_PATH = ""
_SP_TEAM_ALLOWED = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz_-"
_SP_TEAM_MAX_LEN = 32
_SP_TEAM_NUMERIC_ONLY = True

def _sp_team_howto(head):
    """중단 사유 + 학생이 바로 고칠 수 있는 안내를 한 덩어리로 만든다."""
    rule = (
        "1 이상의 정수를 문자열로 입력합니다. 예: 1, 2, 17"
        if _SP_TEAM_NUMERIC_ONLY
        else "영문·숫자·밑줄(_)·하이픈(-) 1~" + str(_SP_TEAM_MAX_LEN) + "자"
    )
    return (
        head
        + "\n"
        + "\n  [고치는 법] 이 셀 맨 위 ★ 상자 안의 한 줄을 이렇게 바꾸세요."
        + '\n      _SP_TEAM = "1"      ← 운영진이 알려준 팀 식별자(숫자)를 따옴표 안에 그대로'
        + "\n  [쓸 수 있는 값] " + rule
        + "\n                 띄어쓰기와 / \\ . : 같은 경로 문자는 파일 이름을 깨뜨려 쓸 수 없습니다."
        + "\n  [왜] 결과 파일 이름이 " + _SP_OUTPUT_PREFIX + "<팀>.json 이고, 채점은 이 이름으로"
        + "\n       어느 팀 답안인지 가립니다. 비워 두면 채점 자체가 되지 않습니다."
    )

def _sp_resolve_team(value):
    """_SP_TEAM 을 검사·정리해 돌려준다. 쓸 수 없는 값이면 RuntimeError 로 즉시 중단."""
    if not isinstance(value, str) or not value.strip():
        raise RuntimeError(_sp_team_howto(
            "★ 팀 식별자(_SP_TEAM)가 비어 있어 실행을 중단했습니다. 결과 파일은 만들지 않았습니다."))
    team = value.strip()
    if _SP_TEAM_NUMERIC_ONLY and not _sp_re.fullmatch(r"[1-9][0-9]*", team):
        raise RuntimeError(_sp_team_howto(
            "★ 팀 식별자(_SP_TEAM)는 운영진이 알려준 숫자여야 합니다. 지금 값: " + repr(value)))
    if len(team) > _SP_TEAM_MAX_LEN:
        raise RuntimeError(_sp_team_howto(
            "★ 팀 식별자(_SP_TEAM)가 너무 깁니다(" + str(len(team)) + "자). 팀 이름이 아니라 짧은 식별자입니다."))
    _bad = _sp_builtins.sorted(
        _sp_builtins.set(c for c in team if c not in _SP_TEAM_ALLOWED and not ("가" <= c <= "힣")))
    if _bad:
        raise RuntimeError(_sp_team_howto(
            "★ 팀 식별자(_SP_TEAM)에 파일 이름으로 쓸 수 없는 문자가 있습니다: "
            + ", ".join(repr(c) for c in _bad) + "   (지금 값: " + repr(value) + ")"))
    return team

_SP_TEAM = _sp_resolve_team(_SP_TEAM)
if any(ord(c) > 127 for c in _SP_TEAM):
    _sp_print("[주의] 팀 식별자에 한글 등 ASCII 밖 문자가 있습니다: " + _SP_TEAM
              + " — 운영진이 알려준 식별자가 맞는지 확인하세요."
              " 한글 파일 이름은 내려받기·올리기 과정에서 자모 표현이 달라져 팀이 어긋날 수 있습니다.",
              flush=True)

_SP_OUTPUT_PATH = _SP_OUTPUT_DIR.rstrip("/") + "/" + _SP_OUTPUT_PREFIX + _SP_TEAM + ".json"
if _SP_EXPECTED_OUTPUT_PATH and (_sp_os_rt.path.basename(_SP_OUTPUT_PATH)
                                 != _sp_os_rt.path.basename(_SP_EXPECTED_OUTPUT_PATH)):
    raise RuntimeError(
        "이 셀은 " + _sp_os_rt.path.basename(_SP_EXPECTED_OUTPUT_PATH) + " 용으로 생성됐는데 "
        + _sp_os_rt.path.basename(_SP_OUTPUT_PATH) + " 로 저장하려 합니다"
        "(_SP_TEAM 을 손으로 고쳤습니까?). 다른 팀으로 돌리려면 --team 을 바꿔 셀을 다시 생성하세요."
    )
_SP_AUTO_DOWNLOAD = True
_SP_QUESTIONS_JSON = (
    "[[\"P01\", \"사업자/단체 카카오계정은 계정 정보에 등록된 담당자 몇 명이 이용할 수 있으며, 다른 사람과 공유하는 것은 허용되나요?\"], [\"P02\", \"회사가 예측하거나 통제할 수 없는 사유로 서비스가 중단된 경우, 복구가 몇 시간 이상 지연되면 회사는 공지사항에 게시하여 알리나요?\"], [\"P03\", \"카카오계정 약관에서 회사가 개별 서비스와 연동하여 카카오계정에서 제공한다고 열거한 '카카오계정 서비스'의 내용 5가지는 각각 무엇인가요?\"], [\"P04\", \"회사가 위치기반서비스의 이용을 제한하거나 중지한 때에는 이용자에게 무엇을 어떤 방법으로 알리나요?\"], [\"P05\", \"회사가 위치정보 수집·이용·제공사실 확인자료를 기록·보존하는 근거는 위치정보의 보호 및 이용 등에 관한 법률 제 몇 조 제 몇 항이며, 그 자료는 어디에 기록되어 몇 개월간 보관되나요?\"], [\"P06\", \"카카오계정이 없는 사람이 통합서비스에 가입하려면 무엇을 먼저 해야 하며, 통합서비스 이용계약은 동의·확인·승낙의 어떤 순서로 체결되나요?\"], [\"P07\", \"서비스 명칭에 '카카오'가 사용되더라도 카카오 통합서비스약관의 '통합서비스'에 포함되지 않는 서비스는 누가 제공하는 서비스이며, 약관은 그 예로 무엇을 들고 있나요?\"], [\"P08\", \"카카오 통합 약관과 세부지침(회사가 정한 서비스의 개별 이용약관·운영정책·규칙 등)의 내용이 충돌하는 경우"
    ", 본 약관이 세부지침보다 우선하여 적용되나요?\"], [\"P09\", \"이용자가 서비스 사용을 중단하거나 카카오계정 및 Daum 아이디를 탈퇴한 이후, 게시물에 관하여 회사에 부여한 라이선스의 효력은 어떻게 되나요?\"], [\"P10\", \"8세 이하의 아동 등의 생명 또는 신체 보호를 위해 보호의무자가 개인위치정보의 이용 또는 제공에 동의하려면 어떤 서류에 무엇을 첨부하여 어디에 제출해야 하며, 그 동의는 어떤 효력을 갖나요?\"]]"
)
_SP_QUESTIONS = [tuple(_x) for _x in _sp_json.loads(_SP_QUESTIONS_JSON)]
_SP_ALLOWED_DOCS = _sp_json.loads("[\"카카오계정 약관\", \"카카오 통합서비스약관\", \"카카오 통합 약관\", \"카카오 위치정보 이용약관\"]")
_SP_PER_Q_TIMEOUT_S = 120
_SP_TRANSPORT = "http"
_SP_HTTP_HOST = "127.0.0.1"
_SP_HTTP_PORT = 8765
_SP_HTTP_STARTUP_TIMEOUT_S = 30
_SP_HTTP_HEALTH_PATH = "/health"
_SP_HTTP_ANSWER_PATH = "/answer"
_SP_PERFORMANCE_REQUESTS = 12
_SP_PERFORMANCE_CONCURRENCY = 2
_SP_PERFORMANCE_REPETITIONS = 3
_SP_PERFORMANCE_WARMUP_REQUESTS = 2

_sp_fn = globals().get("answer_question")
if not callable(_sp_fn):
    raise RuntimeError(
        "팀 코드에 answer_question(question) 함수가 없습니다(규정 ②). 실행을 중단합니다."
    )

_sp_doc_warnings = []
_sp_timeouts = []
_sp_http_server = None
_sp_http_thread = None

class _SpHttpTimeout(Exception):
    """HTTP 요청 시간 초과. 품질 추출에서는 timeout_qids로 기록한다."""

def _sp_http_url(path):
    return "http://" + _SP_HTTP_HOST + ":" + str(_SP_HTTP_PORT) + path

def _sp_http_json(method, path, payload=None, timeout_s=None):
    data = None
    headers = {"Accept": "application/json"}
    if payload is not None:
        data = _sp_json.dumps(payload, ensure_ascii=False).encode("utf-8")
        headers["Content-Type"] = "application/json"
    req = _sp_urlrequest.Request(
        _sp_http_url(path), data=data, headers=headers, method=method
    )
    try:
        with _sp_urlrequest.urlopen(req, timeout=timeout_s or _SP_PER_Q_TIMEOUT_S) as resp:
            raw = resp.read().decode("utf-8")
            if resp.status != 200:
                raise RuntimeError("HTTP " + str(resp.status) + ": " + raw[:500])
    except (_sp_socket.timeout, TimeoutError) as exc:
        raise _SpHttpTimeout(str(timeout_s or _SP_PER_Q_TIMEOUT_S) + "초 안에 응답하지 않았습니다.") from exc
    except _sp_urlerror.HTTPError as exc:
        raw = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError("HTTP " + str(exc.code) + ": " + raw[:500]) from exc
    except _sp_urlerror.URLError as exc:
        if isinstance(exc.reason, (_sp_socket.timeout, TimeoutError)):
            raise _SpHttpTimeout(
                str(timeout_s or _SP_PER_Q_TIMEOUT_S) + "초 안에 응답하지 않았습니다."
            ) from exc
        raise RuntimeError("HTTP 연결 실패: " + str(exc.reason)) from exc
    try:
        return _sp_json.loads(raw)
    except _sp_json.JSONDecodeError as exc:
        raise TypeError("HTTP 응답이 JSON이 아닙니다: " + raw[:500]) from exc

def _sp_start_http_server():
    global _sp_http_server, _sp_http_thread
    _sp_app = globals().get("app")
    if _sp_app is None:
        raise RuntimeError(
            "HTTP 실행 모드에는 전역 FastAPI app과 GET /health, POST /answer가 필요합니다."
        )
    try:
        import threading as _sp_threading
        import uvicorn as _sp_uvicorn
    except ImportError as exc:
        raise RuntimeError(
            "HTTP 실행 모드에는 fastapi와 uvicorn이 필요합니다. 팀 설치 목록에 추가하세요."
        ) from exc
    _sp_config = _sp_uvicorn.Config(
        _sp_app,
        host=_SP_HTTP_HOST,
        port=_SP_HTTP_PORT,
        workers=1,
        log_level="warning",
        access_log=False,
    )
    _sp_http_server = _sp_uvicorn.Server(_sp_config)
    _sp_http_thread = _sp_threading.Thread(
        target=_sp_http_server.run, name="ktb-fastapi", daemon=True
    )
    _sp_http_thread.start()
    _sp_deadline = _sp_time.time() + _SP_HTTP_STARTUP_TIMEOUT_S
    _sp_last = None
    while _sp_time.time() < _sp_deadline:
        if not _sp_http_thread.is_alive():
            raise RuntimeError("FastAPI 서버가 준비되기 전에 종료됐습니다.")
        try:
            health = _sp_http_json("GET", _SP_HTTP_HEALTH_PATH, timeout_s=1)
            if isinstance(health, dict):
                _sp_print("[서버] FastAPI /health 준비 완료: " + _sp_http_url(_SP_HTTP_HEALTH_PATH))
                return
        except Exception as exc:
            _sp_last = exc
        _sp_time.sleep(0.2)
    _sp_stop_http_server()
    raise RuntimeError(
        "FastAPI 서버가 " + str(_SP_HTTP_STARTUP_TIMEOUT_S)
        + "초 안에 준비되지 않았습니다: " + str(_sp_last)
    )

def _sp_stop_http_server():
    if _sp_http_server is not None:
        _sp_http_server.should_exit = True
    if _sp_http_thread is not None and _sp_http_thread.is_alive():
        _sp_http_thread.join(timeout=5)

def _sp_invoke(question):
    if _SP_TRANSPORT == "http":
        return _sp_http_json(
            "POST", _SP_HTTP_ANSWER_PATH, {"question": question},
            timeout_s=_SP_PER_Q_TIMEOUT_S,
        )
    return _sp_call_with_timeout(_sp_fn, question, _SP_PER_Q_TIMEOUT_S)

if _SP_TRANSPORT == "http":
    _sp_start_http_server()

_sp_env_warnings = []
for _sp_d in ("/content/drive", "/content/gdrive", "/gdrive"):
    if _sp_os_rt.path.ismount(_sp_d):
        _sp_env_warnings.append(_sp_d + " 가 마운트되어 있습니다")
if _sp_env_warnings:
    _sp_print("", flush=True)
    _sp_print("!" * 86, flush=True)
    _sp_print("[규정 ③ 경고] 이 세션은 운영진 실행 환경과 다릅니다.", flush=True)
    for _sp_w in _sp_env_warnings:
        _sp_print("  · " + _sp_w, flush=True)
    _sp_print("  운영진은 드라이브가 연결되지 않은 새 세션에서 실행합니다. 드라이브에 둔 약관·인덱스를", flush=True)
    _sp_print("  읽고 있다면 본선에서 전량 실패합니다. 약관은 실행 중 내려받거나 셀 안에 포함하세요.", flush=True)
    _sp_print("  확인 방법: 새 노트북을 열어 코드와 이 셀만 붙여 넣고 실행해 보세요.", flush=True)
    _sp_print("!" * 86, flush=True)
    _sp_print("", flush=True)

class _SpTimeout(BaseException):
    """문항 단위 시간 초과.

    **BaseException 을 상속하는 것이 핵심이다.** 팀 코드가 `try/except Exception` 으로
    넓게 감싸는 일은 흔한데, Exception 을 상속하면 그 handler 가 시간 초과를 삼켜
    상한이 무력화된다(그대로 다음 루프를 돌며 계속 매달린다).
    """

def _sp_call_with_timeout(fn, arg, seconds):
    """SIGALRM 으로 문항 호출에 상한을 건다.

    메인 스레드가 아니거나 SIGALRM 이 없는 환경(윈도 등)에서는 signal 설정이
    실패하므로, 그때는 상한 없이 그대로 호출한다 — 상한을 못 걸었다고 해서
    채점 자체를 포기하는 편이 더 나쁘다.

    웹 Colab 셀은 IPython 이 메인 스레드에서 실행하므로 정상 동작한다.
    """
    if not seconds or seconds <= 0:
        return fn(arg)
    _sp_secs = max(1, int(seconds))     # alarm() 은 정수만 받는다. 0 은 '취소' 라 최소 1초.

    def _sp_on_alarm(signum, frame):
        raise _SpTimeout(str(_sp_secs) + "초 안에 응답하지 않았습니다.")

    try:
        _sp_prev = _sp_signal.signal(_sp_signal.SIGALRM, _sp_on_alarm)
        _sp_signal.alarm(_sp_secs)
    except (ValueError, AttributeError, OSError):
        return fn(arg)          # 상한을 걸 수 없는 환경 — 그대로 실행
    try:
        return fn(arg)
    finally:
        _sp_signal.alarm(0)
        try:
            _sp_signal.signal(_sp_signal.SIGALRM, _sp_prev)
        except Exception:
            pass

def _sp_json_safe_art(art):
    """조번호를 JSON 으로 쓸 수 있는 값으로. 표기는 최대한 원본을 살린다.

    **여기서 흡수하지 않으면 30문항을 다 돌린 뒤 파일 저장에서 터진다.**
    일부 수치 라이브러리의 정수형은 dict 도 아니고 2원소 검사도 통과하지만
    json.dump 가 거부한다. 이 값을 흡수하지 않으면
    실패 시점이 맨 끝이라 GPU 시간을 다 쓰고 결과 파일이 없는 최악의 형태가 된다.

    '제7조' 같은 문자열은 그대로 둔다 — 채점기 _art_no 가 정수로 읽는다.
    """
    if isinstance(art, bool):        # bool 은 int 의 하위형이라 먼저 걸러 낸다
        return str(art)
    if isinstance(art, (int, str)):
        return art
    try:                              # np.int64 등 정수로 볼 수 있는 것
        return int(art)
    except (TypeError, ValueError):
        return str(art)

def _sp_norm_doc(x):
    """문서명 대조용 정규화 — NFC 통일 + 공백 전부 제거.

    ⚠️ 채점기 judge_service/engine/objective.py 의 `norm_doc` 과 **같은 규칙이어야 한다.**
    러너는 Colab 셀이라 judge_service 를 import 할 수 없어 규칙을 여기에 복제해 둔다.
    한쪽만 바뀌어 어긋나면 곧바로 오탐이 난다 — 예전에 러너가 완전 일치로 대조하던 때
    '카카오계정약관'·'카카오 계정 약관' 은 실제 채점 MRR 이 1.00 인데도 규정 ④ 위반 경고를
    맞았다. 팀은 없는 문제를 고치러 다니고(자가 확인표가 n_doc_violations == 0 을 요구한다),
    정상 팀이 경고를 맞기 시작하면 아무도 경고를 안 보게 된다.
    두 구현의 일치는 submission_pipeline/tests/test_doc_name_normalization.py 가 고정한다.
    """
    return _sp_re.sub(r"\s+", "", _sp_unicodedata.normalize("NFC", str(x)))

_SP_ALLOWED_DOCS_NORM = _sp_builtins.set(_sp_norm_doc(_d) for _d in _SP_ALLOWED_DOCS)

def _sp_normalize_retrieved(qid, value):
    """retrieved 를 근거순 [[문서명, 조번호], ...] 1~4개로 정규화."""
    if not isinstance(value, (list, tuple)):
        raise TypeError(qid + ": retrieved 는 목록이어야 합니다. (실제: " + type(value).__name__ + ")")
    out = []
    for item in value:
        if isinstance(item, dict) and "doc" in item and "article_no" in item:
            doc, art = item["doc"], item["article_no"]
        elif isinstance(item, (list, tuple)) and len(item) == 2:
            doc, art = item
        else:
            raise TypeError(qid + ": retrieved 항목은 [문서명, 조번호] 2원소여야 합니다. (실제: " + repr(item) + ")")
        doc = str(doc)
        if _SP_ALLOWED_DOCS_NORM and _sp_norm_doc(doc) not in _SP_ALLOWED_DOCS_NORM:
            _sp_doc_warnings.append({"qid": qid, "doc": doc})
        out.append([doc, _sp_json_safe_art(art)])
    if not 1 <= len(out) <= 4:
        raise ValueError(
            qid + ": retrieved 는 실제 답변 근거를 관련도 순으로 1~4개 반환해야 합니다. "
            "(실제: " + str(len(out)) + "개)"
        )
    return out

_sp_answers = []
_sp_errors = []
_sp_total = len(_SP_QUESTIONS)
_sp_print(
    "\n========== " + "공개" + " " + str(_sp_total)
    + "문항 실행 · " + _SP_TEAM + "팀 ==========",
    flush=True,
)
_sp_t0 = _sp_time.time()

for _sp_i, (_sp_qid, _sp_q) in enumerate(_SP_QUESTIONS, 1):
    _sp_print("[" + str(_sp_i).zfill(2) + "/" + str(_sp_total) + "] " + _sp_qid + " 실행 중 ...", flush=True)
    _sp_started = _sp_time.time()
    try:
        _sp_out = _sp_invoke(_sp_q)
        if not isinstance(_sp_out, dict):
            raise TypeError(_sp_qid + ": answer_question() 은 딕셔너리를 반환해야 합니다. (실제: "
                            + type(_sp_out).__name__ + ")")
        _sp_retrieved = _sp_normalize_retrieved(_sp_qid, _sp_out.get("retrieved"))
        _sp_answer = _sp_out.get("answer")
        if not isinstance(_sp_answer, str):
            raise TypeError(_sp_qid + ": answer 는 문자열이어야 합니다. (실제: "
                            + type(_sp_answer).__name__ + ")")
        _sp_answers.append({"qid": _sp_qid, "retrieved": _sp_retrieved, "answer": _sp_answer})
    except (_SpTimeout, _SpHttpTimeout) as _sp_exc:  # 한 문항이 세션 전체를 잡아먹지 않도록 끊는다.
        _sp_msg = "Timeout: " + str(_sp_exc)
        _sp_timeouts.append(_sp_qid)
        _sp_errors.append({"qid": _sp_qid, "error": _sp_msg})
        _sp_answers.append({"qid": _sp_qid, "retrieved": [], "answer": "", "error": _sp_msg})
        _sp_print("[시간초과] " + _sp_qid + " — " + _sp_msg, flush=True)
    except Exception as _sp_exc:  # 한 문항 실패로 30문항 전체를 잃지 않는다.
        _sp_msg = type(_sp_exc).__name__ + ": " + str(_sp_exc)
        _sp_errors.append({"qid": _sp_qid, "error": _sp_msg})
        _sp_answers.append({"qid": _sp_qid, "retrieved": [], "answer": "", "error": _sp_msg})
        _sp_print("[오류] " + _sp_qid + " — " + _sp_msg, flush=True)
        _sp_traceback.print_exc()
    finally:
        _sp_print("      (" + str(round(_sp_time.time() - _sp_started, 1)) + "s)", flush=True)

_sp_performance = None
if _SP_TRANSPORT == "http" and _SP_PERFORMANCE_REQUESTS > 0:
    from concurrent.futures import ThreadPoolExecutor as _SpThreadPoolExecutor

    def _sp_perf_one(index):
        _qid, _question = _SP_QUESTIONS[index % len(_SP_QUESTIONS)]
        started = _sp_time.perf_counter()
        try:
            value = _sp_http_json(
                "POST", _SP_HTTP_ANSWER_PATH, {"question": _question},
                timeout_s=_SP_PER_Q_TIMEOUT_S,
            )
            ok = (
                isinstance(value, dict)
                and isinstance(value.get("answer"), str)
                and isinstance(value.get("retrieved"), (list, tuple))
            )
            return {
                "ok": ok,
                "qid": _qid,
                "latency_s": round(_sp_time.perf_counter() - started, 4),
                "error": None if ok else "invalid_schema",
            }
        except Exception as exc:
            return {
                "ok": False,
                "qid": _qid,
                "latency_s": round(_sp_time.perf_counter() - started, 4),
                "error": type(exc).__name__ + ": " + str(exc),
            }

    def _sp_percentile(values, ratio):
        if not values:
            return None
        pos = min(len(values) - 1, max(0, int((len(values) - 1) * ratio)))
        return round(values[pos], 4)

    def _sp_median(values):
        values = sorted(values)
        if not values:
            return None
        middle = len(values) // 2
        if len(values) % 2:
            return values[middle]
        return (values[middle - 1] + values[middle]) / 2

    def _sp_perf_round(n_requests, repetition):
        started = _sp_time.perf_counter()
        with _SpThreadPoolExecutor(max_workers=max(1, _SP_PERFORMANCE_CONCURRENCY)) as pool:
            rows = list(pool.map(_sp_perf_one, range(n_requests)))
        wall_s = _sp_time.perf_counter() - started
        ok_rows = [row for row in rows if row["ok"]]
        latencies = sorted(row["latency_s"] for row in ok_rows)
        return {
            "repetition": repetition,
            "transport": "http",
            "requests": n_requests,
            "concurrency": _SP_PERFORMANCE_CONCURRENCY,
            "success": len(ok_rows),
            "fail": len(rows) - len(ok_rows),
            "success_rate": round(len(ok_rows) / len(rows), 4),
            "throughput_rps": round(len(ok_rows) / wall_s, 4) if wall_s else 0.0,
            "wall_s": round(wall_s, 4),
            "p50_latency_s": _sp_percentile(latencies, 0.50),
            "p95_latency_s": _sp_percentile(latencies, 0.95),
            "errors": [row for row in rows if not row["ok"]],
        }

    _sp_warmup = None
    if _SP_PERFORMANCE_WARMUP_REQUESTS > 0:
        _sp_print(
            "[성능] 워밍업 " + str(_SP_PERFORMANCE_WARMUP_REQUESTS) + "요청 실행 중 ...",
            flush=True,
        )
        _sp_warmup = _sp_perf_round(_SP_PERFORMANCE_WARMUP_REQUESTS, 0)

    _sp_perf_samples = []
    for _sp_repetition in range(1, _SP_PERFORMANCE_REPETITIONS + 1):
        _sp_print(
            "[성능] 측정 " + str(_sp_repetition) + "/"
            + str(_SP_PERFORMANCE_REPETITIONS) + " 실행 중 ...",
            flush=True,
        )
        _sp_perf_samples.append(
            _sp_perf_round(_SP_PERFORMANCE_REQUESTS, _sp_repetition)
        )

    _sp_success_median = _sp_median([row["success"] for row in _sp_perf_samples])
    _sp_fail_median = _sp_median([row["fail"] for row in _sp_perf_samples])
    _sp_p50_values = [
        row["p50_latency_s"] for row in _sp_perf_samples
        if row["p50_latency_s"] is not None
    ]
    _sp_p95_values = [
        row["p95_latency_s"] for row in _sp_perf_samples
        if row["p95_latency_s"] is not None
    ]
    _sp_performance = {
        "version": 2,
        "transport": "http",
        "requests": _SP_PERFORMANCE_REQUESTS,
        "concurrency": _SP_PERFORMANCE_CONCURRENCY,
        "success": int(_sp_success_median),
        "fail": int(_sp_fail_median),
        "success_rate": round(_sp_median(
            [row["success_rate"] for row in _sp_perf_samples]
        ), 4),
        "throughput_rps": round(_sp_median(
            [row["throughput_rps"] for row in _sp_perf_samples]
        ), 4),
        "wall_s": round(_sp_median(
            [row["wall_s"] for row in _sp_perf_samples]
        ), 4),
        "p50_latency_s": (
            round(_sp_median(_sp_p50_values), 4) if _sp_p50_values else None
        ),
        "p95_latency_s": (
            round(_sp_median(_sp_p95_values), 4) if _sp_p95_values else None
        ),
        "errors": [
            dict(error, repetition=sample["repetition"])
            for sample in _sp_perf_samples
            for error in sample["errors"]
        ],
        "summary_method": "median",
        "protocol": {
            "requests_per_run": _SP_PERFORMANCE_REQUESTS,
            "concurrency": _SP_PERFORMANCE_CONCURRENCY,
            "warmup_requests": _SP_PERFORMANCE_WARMUP_REQUESTS,
            "repetitions": _SP_PERFORMANCE_REPETITIONS,
        },
        "samples": _sp_perf_samples,
    }
    if _sp_warmup is not None:
        _sp_performance["warmup"] = _sp_warmup
    _sp_print(
        "[성능] closed-loop 중앙값 · "
        + str(_SP_PERFORMANCE_REQUESTS) + "요청 × "
        + str(_SP_PERFORMANCE_REPETITIONS) + "회 · 동시성 "
        + str(_SP_PERFORMANCE_CONCURRENCY) + " · 대표 성공 "
        + str(_sp_performance["success"]) + " · "
        + str(_sp_performance["throughput_rps"]) + " req/s · p95 "
        + str(_sp_performance["p95_latency_s"]) + "s",
        flush=True,
    )

_sp_stop_http_server()

_sp_submission = {"team": _SP_TEAM, "answers": _sp_answers}
if _sp_doc_warnings or _sp_timeouts or _sp_env_warnings or _sp_performance:
    _sp_submission["meta"] = {"doc_name_violations": _sp_doc_warnings,
                              "timeout_qids": _sp_timeouts,
                              "env_warnings": _sp_env_warnings,
                              "transport": _SP_TRANSPORT}
    if _sp_performance:
        _sp_submission["meta"]["performance"] = _sp_performance
_sp_text = _sp_json.dumps(_sp_submission, ensure_ascii=False, indent=2, default=str)
with _sp_open(_SP_OUTPUT_PATH, "w", encoding="utf-8") as _sp_f:
    _sp_f.write(_sp_text)

_sp_print("[완료] " + str(len(_sp_answers)) + "문항 저장: " + _SP_OUTPUT_PATH
      + "  (총 " + str(round(_sp_time.time() - _sp_t0, 1)) + "s)", flush=True)
if _sp_errors:
    _sp_print("[경고] 실패 문항 " + str(len(_sp_errors)) + "건: "
          + ", ".join(_e["qid"] for _e in _sp_errors), flush=True)
if _sp_doc_warnings:
    _sp_print("[경고] 규정 ④ 위반 — 허용 목록 밖 문서명 " + str(len(_sp_doc_warnings)) + "건: "
          + ", ".join(sorted(set(_w["doc"] for _w in _sp_doc_warnings)))
          + "  → 해당 항목은 검색 점수가 0으로 채점됩니다. 허용(띄어쓰기 차이는 무관): "
          + ", ".join(_SP_ALLOWED_DOCS), flush=True)

if _SP_AUTO_DOWNLOAD:
    try:
        from google.colab import files as _sp_files
        _sp_files.download(_SP_OUTPUT_PATH)
        _sp_print("[다운로드] 브라우저 다운로드를 시작했습니다: " + _SP_OUTPUT_PATH, flush=True)
    except Exception as _sp_dl_exc:
        _sp_print("[다운로드] 자동 다운로드 실패(" + type(_sp_dl_exc).__name__ + ": " + str(_sp_dl_exc)
                  + ") — 좌측 파일 탭에서 " + _SP_OUTPUT_PATH + " 를 직접 내려받으세요.", flush=True)

_sp_print("SUBMISSION_RUNNER_DONE " + _sp_json.dumps(
    {"team": _SP_TEAM, "output_path": _SP_OUTPUT_PATH, "n_answers": len(_sp_answers),
     "n_errors": len(_sp_errors), "failed_qids": [_e["qid"] for _e in _sp_errors],
     "n_doc_violations": len(_sp_doc_warnings), "timeout_qids": _sp_timeouts,
     "env_warnings": _sp_env_warnings, "transport": _SP_TRANSPORT,
     "performance": _sp_performance},
    ensure_ascii=False), flush=True)


[서버] FastAPI /health 준비 완료: http://127.0.0.1:8765/health

========== 공개 10문항 실행 · 8팀 ==========
[01/10] P01 실행 중 ...
      (10.6s)
[02/10] P02 실행 중 ...
      (9.4s)
[03/10] P03 실행 중 ...
      (32.4s)
[04/10] P04 실행 중 ...
      (8.5s)
[05/10] P05 실행 중 ...
      (7.6s)
[06/10] P06 실행 중 ...
      (7.2s)
[07/10] P07 실행 중 ...
      (9.2s)
[08/10] P08 실행 중 ...
      (5.4s)
[09/10] P09 실행 중 ...
      (26.4s)
[10/10] P10 실행 중 ...
      (10.0s)
[성능] 워밍업 2요청 실행 중 ...
[성능] 측정 1/3 실행 중 ...
[성능] 측정 2/3 실행 중 ...
